In [1]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub

from langchain.tools import tool
import requests

In [2]:
from langchain.agents import create_react_agent, AgentExecutor

In [3]:
# =======================
# LOAD ENV VARIABLES
# ======================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [4]:
search_tool = TavilySearchResults(max_results=3)

In [17]:
# =======================
# CUSTOM TOOL (API)
# ======================
@tool
def get_weather_data(city: str):
    """
    Fetch current weather information for a city
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )
    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"city: {city}\n"
        f"Temperature: {data['current']['temperature']}⁰C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity; {data['current']['humidity']}%"
    )

In [18]:
result = search_tool.invoke("Give me the latest news of AI?")
result


[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'Artificial Intelligence\n\nApril 11, 2024\n\n### UK and Canada sign AI compute agreement\n\nArtificial Intelligence\n\nJanuary 31, 2024\n\n#### Machine Learning\n\n### MIT AI forecasts extreme weather without historical data\n\nEnvironment & Sustainability\n\nAugust 25, 2026\n\n### Samsung health AI models analyse wearable biosignal data\n\nHealthcare & Wellness AI\n\nAugust 14, 2026\n\n### Why biological data matters more in AI drug discovery\n\nAI in Action\n\nAugust 3, 2026\n\n#### Enterprise\n\n### OpenAI president urges enterprises to hasten AI security defences\n\nCybersecurity AI\n\nAugust 18, 2026\n\n### Okta targets AI agent token costs with MCP scoping\n\nData Engineering & MLOps\n\nAugust 13, 2026\n\n### Red Hat, NVIDIA, IBM back project turning AI policy into code\n\nGovernance, Regulation & Policy\n\nAugust 4, 2026 [...] Explore More\n\n# Hershey applies AI across its supply chain operations\n\n# With hi

In [19]:
# =======================
# LLM
# ======================

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=OPENAI_API_KEY
)

In [20]:
response = llm.invoke("what year is it ")
response

AIMessage(content='It is currently 2021.', response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 12, 'total_tokens': 19, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-7a12dd07-cc91-40c8-aef3-d96694e7e703-0')

In [21]:
# =======================
# PROMPT
# ======================
prompt = hub.pull("hwchase17/react")
prompt


/opt/anaconda3/envs/langagent/lib/python3.11/site-packages/langchain/hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [22]:
# =======================
# PROMPT
# ======================

tools = [search_tool, get_weather_data]


In [23]:
# =======================
# CREATE AGENT
# ======================
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

# =======================
# AGENT EXECUTOR
# ======================
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [24]:
# =======================
# RUN
# ======================
response = agent_executor.invoke({
    "input": (
"Find the capital of Burundi"
"and then find its current weather"
    )
})





> Entering new AgentExecutor chain...
I should first search for the capital of Burundi and then use the weather data tool to find its current weather.
Action: tavily_search_results_json
Action Input: "capital of Burundi"[{'url': 'https://www.youtube.com/watch?v=DoNDxjs2mng', 'content': '### Description\n26222 views\nPosted: 2 Sep 2024\nGitega formerly Kitega, is the political capital of Burundi. Located in the centre of the country, in the Burundian central plateau roughly 62 kilometres east of Bujumbura, the largest city and former political capital, Gitega is also the second largest city and former royal capital of the Kingdom of Burundi until its abolition in 1966. \nIn late December 2018, Burundian president Pierre Nkurunziza announced that he would follow on a 2007 promise to return Gitega its former political capital status, with Bujumbura remaining as economic capital and centre of commerce. A vote in the Parliament of Burundi made the change official on 16 January 2019, with 

In [25]:
print(response["output"])

The capital of Burundi is Gitega, and the current weather there is 28⁰C with partly cloudy skies and 30% humidity.
